# Rung 2 — fit the **simple** Bayesian model to real data

Direct next step after `earth_laquitaine_gen_ext.ipynb`. That notebook was *generative*: build one Gaussian posterior, read one MAP estimate, sweep. Here we take the **same Gaussian observer** and **fit it to real human reports**, changing exactly one thing: instead of *producing* estimates we ask *how probable the subject's actual estimate is*, and tune the parameters to maximise that.

Deliberately kept minimal — still **non-circular** (straight direction axis, no von Mises, no `kappa`) and **static only** (one fixed prior, no online/warm-up). The whole ladder:

1. `earth_laquitaine_gen_ext.ipynb` — Gaussian, generative *(done)*
2. **this notebook** — Gaussian, fitted to real data
3. add the circle — von Mises, generative *(next)*
4. `earth_laquitaine_circular_fit_ext.ipynb` — circular, fitted to real data *(done)*
5. richer models — online prior, warm-up, motor noise, AIC comparison

**The one idea that keeps this simple.** For a Gaussian prior × Gaussian likelihood the MAP estimate is *analytic* (Tutorial 2). So we never need a grid or marginalisation — `P(estimate | stimulus)` is itself a Gaussian we can write in one line and hand to an optimiser.

In [ ]:
# @title Dependencies + data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.optimize import minimize
import os, requests

rcParams['figure.figsize'] = [8, 4]
rcParams['font.size'] = 11
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

URL = "https://github.com/steevelaquitaine/projInference/raw/gh-pages/data/csv/data01_direction4priors.csv"
CSV = "data01_direction4priors.csv"
if not os.path.exists(CSV):
    r = requests.get(URL)
    assert r.status_code == 200, "download failed; contact steeve.laquitaine@epfl.ch"
    open(CSV, "wb").write(r.content)

data = pd.read_csv(CSV)
print(data.shape)
data[["subject_id", "prior_std", "motion_coherence", "motion_direction"]].head()

## Step 1 — the analytic Gaussian observer

From `earth_laquitaine_gen_ext.ipynb`: the observer gets a noisy measurement $m \sim \mathcal N(\theta, \sigma_s^2)$ of the true direction $\theta$, and holds a Gaussian prior $\mathcal N(\mu_0, \sigma_p^2)$ with $\mu_0 = 225°$. The MAP estimate is the precision-weighted average

$$\hat\theta(m) = w\,m + (1-w)\,\mu_0, \qquad w = \frac{\sigma_p^2}{\sigma_s^2 + \sigma_p^2}.$$

Because $m$ is Gaussian, the *estimate itself* is Gaussian across trials of the same stimulus:

$$\hat\theta \mid \theta \ \sim\ \mathcal N\!\big(\underbrace{w\,\theta + (1-w)\,\mu_0}_{\text{predicted mean}},\ \underbrace{w^2\sigma_s^2}_{\text{predicted variance}}\big).$$

The prior weight $w$ falls as coherence falls (larger $\sigma_s$), so low-coherence estimates get pulled harder toward 225° — the whole Laquitaine effect, in closed form. `predict(...)` below returns that `(mean, sd)`; no grid needed.

In [ ]:
# @title The observer as a one-liner
MU0 = 225.0   # prior mean (fixed by the experiment)

def prior_weight(sig_s, sig_p):
    """how much the estimate follows the stimulus vs the prior (0..1)."""
    return sig_p ** 2 / (sig_s ** 2 + sig_p ** 2)

def predict(theta_deg, sig_s, sig_p):
    """Gaussian P(estimate | stimulus): returns (predicted mean, predicted sd)."""
    w = prior_weight(sig_s, sig_p)
    mean = w * theta_deg + (1 - w) * MU0
    sd = w * sig_s
    return mean, sd

# quick look: stronger prior / weaker signal -> estimate sits closer to 225
for sig_s in [10, 30, 60]:
    m, sd = predict(160.0, sig_s, sig_p=25.0)
    print(f"sig_s={sig_s:>3}  w={prior_weight(sig_s,25):.2f}  "
          f"stimulus 160 -> predicted estimate {m:.0f} deg (pulled toward 225)")

## Step 2 — one subject, one prior block

Same slice used throughout the project: **subject 1**, the **80° prior block**, its three coherences. We only need three arrays: stimulus direction, the subject's estimate (converted from the (x, y) unit vector to degrees), and a coherence label per trial.

In [ ]:
# @title Build arrays for subject 1, prior_std = 80
SUBJECT_ID = 1
PRIOR_STD = 80

sub = data[(data.subject_id == SUBJECT_ID) & (data.prior_std == PRIOR_STD)].copy()
sub = sub.dropna(subset=["estimate_x", "estimate_y"])

Sdeg = sub.motion_direction.values.astype(float)                                   # stimulus
Edeg = np.degrees(np.arctan2(sub.estimate_y.values, sub.estimate_x.values)) % 360  # subject estimate
coh = sub.motion_coherence.values
coh_levels = np.sort(np.unique(coh))
C = np.searchsorted(coh_levels, coh)                                               # 0,1,2

print(f"{len(Sdeg)} trials | coherences {coh_levels} | stimuli {np.unique(Sdeg)}")

# sanity: raw bias toward 225 already visible, strongest at low coherence
for c in range(3):
    m = C == c
    print(f"{int(coh_levels[c]*100):>2}% coh: mean estimate {Edeg[m].mean():6.1f} deg "
          f"(mean stimulus {Sdeg[m].mean():.1f})")

## Step 3 — the likelihood and the fit

Free parameters: one sensory width per coherence $\sigma_s^{(0,1,2)}$ and one prior width $\sigma_p$ — **four numbers**. The negative log likelihood just sums the Gaussian log-density of each observed estimate under `predict(...)`:

$$\mathrm{NLL} = -\sum_t \log \mathcal N\!\big(\hat\theta_t;\ \text{mean}_t,\ \text{sd}_t\big).$$

Widths must stay positive, so we fit their logs and exponentiate inside `unpack`. That is the entire fitting machinery — no marginalisation, no AIC yet.

In [ ]:
# @title NLL + fit (static Gaussian, analytic)
def unpack(par):
    sig_s = np.exp(par[:3])   # sensory width per coherence
    sig_p = np.exp(par[3])    # prior width
    return sig_s, sig_p

def gauss_logpdf(x, mean, sd):
    return -0.5 * np.log(2 * np.pi * sd ** 2) - 0.5 * ((x - mean) / sd) ** 2

def nll(par, Sdeg, Edeg, C):
    sig_s, sig_p = unpack(par)
    ll = 0.0
    for c in range(3):
        m = C == c
        mean, sd = predict(Sdeg[m], sig_s[c], sig_p)
        ll += gauss_logpdf(Edeg[m], mean, sd).sum()
    return -ll

def fit(Sdeg, Edeg, C, x0):
    return minimize(nll, x0, args=(Sdeg, Edeg, C), method="Nelder-Mead",
                    options={"maxiter": 2000, "xatol": 1e-3, "fatol": 1e-3})

def show(par):
    sig_s, sig_p = unpack(par)
    return f"sig_s={np.round(sig_s,1)}  sig_p={sig_p:.1f}"

x0 = np.log([40.0, 25.0, 15.0, 30.0])   # rough guess: sensory width falls with coherence
print("start:", show(x0))

## Step 4 — sanity check: recover known parameters

Before trusting a fit on real data, confirm the fitter can recover parameters it generated itself (the Model-Fitting rule, W1D2). Simulate estimates from `predict(...)` with **known** widths on the real stimulus sequence, then fit and compare.

In [ ]:
# @title Parameter recovery
rng = np.random.default_rng(0)
true_par = np.log([45.0, 25.0, 13.0, 28.0])
sig_s_t, sig_p_t = unpack(true_par)

E_syn = np.empty(len(Sdeg))
for c in range(3):
    m = C == c
    mean, sd = predict(Sdeg[m], sig_s_t[c], sig_p_t)
    E_syn[m] = rng.normal(mean, sd)

res_syn = fit(Sdeg, E_syn, C, x0)
print("true     :", show(true_par))
print("recovered:", show(res_syn.x))

## Step 5 — fit the real data

Now the same fit on the subject's actual reports. Read out the fitted widths and the implied **prior weight** $w$ per coherence — the single number saying how much the prior wins at each signal strength.

In [ ]:
# @title Fit subject 1 and report
res = fit(Sdeg, Edeg, C, x0)
sig_s, sig_p = unpack(res.x)

print("fitted:", show(res.x), f"   NLL={res.fun:.1f}\n")
print(f"prior width sig_p = {sig_p:.1f} deg\n")
print(f"{'coherence':>10}{'sig_s':>8}{'prior weight w':>16}")
for c in range(3):
    w = prior_weight(sig_s[c], sig_p)
    print(f"{int(coh_levels[c]*100):>9}%{sig_s[c]:>8.1f}{w:>16.2f}")
print("\nlow coherence -> larger sig_s -> smaller w -> estimate pulled harder to 225")

## Step 6 — predicted vs observed

The payoff plot. Per coherence, overlay the model's predicted mean estimate (line) on the subject's binned data (dots). The red dashed line is *veridical* (estimate = stimulus); the grey dotted line is the prior at 225°. A good fit bows away from veridical toward 225°, more so at low coherence.

In [ ]:
# @title Predicted vs observed mean estimate
plt.figure(figsize=(12, 4))
for c in range(3):
    m = C == c
    stims = np.unique(Sdeg[m])
    data_mean = np.array([Edeg[m & (Sdeg == s)].mean() for s in stims])
    pred_mean = predict(stims, sig_s[c], sig_p)[0]

    ax = plt.subplot(1, 3, c + 1)
    ax.plot(stims, data_mean, "o", color=[0.5, 0, 0], label="data")
    ax.plot(stims, pred_mean, "-", color="k", label="model")
    ax.plot([stims.min(), stims.max()], [stims.min(), stims.max()], "r--", lw=1)
    ax.axhline(MU0, color="gray", ls=":")
    ax.set_title(f"{int(coh_levels[c]*100)}% coherence  (w={prior_weight(sig_s[c],sig_p):.2f})")
    ax.set_xlabel("stimulus dir (deg)")
    if c == 0:
        ax.set_ylabel("mean estimate (deg)"); ax.legend()
plt.tight_layout(); plt.show()

## Where this goes next

- **What we did.** Took the generative Gaussian observer from `earth_laquitaine_gen_ext.ipynb` and fit its two width parameters to a real subject by maximising the likelihood of their estimates. Everything stayed analytic and non-circular.
- **Rung 3 — add the circle.** Directions wrap (359° ≈ 1°); a straight-line Gaussian mishandles anything near 0/360. Swap $\mathcal N \to$ von Mises and $\sigma \to \kappa\ (\approx 1/\sigma^2)$. The MAP is no longer a clean weighted average, so we move to the grid + marginalisation approach — that is `earth_laquitaine_circular_fit_ext.ipynb`.
- **Rung 5 — richer models.** Add report/motor noise to the predicted sd, a lapse floor, and the **online** (drifting prior) and **warm-up** variants, then compare with AIC. The heavy Gaussian version of that machinery already exists in `earth_laquitaine_fit_ext.ipynb` if you want to see the full comparison without the circle.